# Notebook 1 — Análisis Exploratorio de Datos (EDA)
**Dataset:** Bank Marketing — UCI Machine Learning Repository
**Objetivo:** Predecir si un cliente suscribirá un depósito a plazo (variable `y`)
**Autor:** Jonathan Sánchez Pesantes

## 1. Configuración e Ingesta

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

# Cargar tabla
df = spark.table("bank_additional_full")

# Limpiar nombres de columnas (puntos generan conflictos en Spark)
for col_name in df.columns:
    if "." in col_name:
        df = df.withColumnRenamed(col_name, col_name.replace(".", "_"))

print(f"Filas: {df.count()} | Columnas: {len(df.columns)}")
df.printSchema()

Filas: 41188 | Columnas: 21
root
 |-- age: long (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- duration: long (nullable = true)
 |-- campaign: long (nullable = true)
 |-- pdays: long (nullable = true)
 |-- previous: long (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- emp_var_rate: double (nullable = true)
 |-- cons_price_idx: double (nullable = true)
 |-- cons_conf_idx: double (nullable = true)
 |-- euribor3m: double (nullable = true)
 |-- nr_employed: double (nullable = true)
 |-- y: string (nullable = true)



## 2. Vista General del Dataset

In [0]:
# Estadísticas descriptivas
display(df.describe())

summary,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
count,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188
mean,40.02406040594348,null,null,null,null,null,null,null,null,null,258.2850101971448,2.567592502670681,962.4754540157328,0.17296299893172767,null,0.0818855006319146,93.57566436831263,-40.50260027191399,3.621290812858068,5167.035910941844,null
stddev,10.421249980934048,null,null,null,null,null,null,null,null,null,259.2792488364648,2.770013542902328,186.9109073447418,0.4949010798392896,null,1.5709597405170423,0.5788400489558622,4.628197856175059,1.7344474048512686,72.2515276682911,null
min,17,admin.,divorced,basic.4y,no,no,no,cellular,apr,fri,0,1,0,0,failure,-3.4,92.201,-50.8,0.634,4963.6,no
max,98,unknown,unknown,unknown,yes,yes,yes,telephone,sep,wed,4918,56,999,7,success,1.4,94.767,-26.9,5.045,5228.1,yes


## 3. Variable Objetivo — Desbalance de Clases

In [0]:
display(
    df.groupBy("y")
    .count()
    .withColumn("porcentaje", round(col("count") / df.count() * 100, 2))
    .orderBy("count", ascending=False)
)



y,count,porcentaje
no,36548,88.73
yes,4640,11.27


> **Hallazgo:** Dataset fuertemente desbalanceado — 88.7% no suscribe vs 11.3% que sí.
> Esto debe tratarse antes de modelar para evitar que el modelo aprenda a predecir siempre "no".

## 4. Tasa de Conversión por Segmento

In [0]:
# Por tipo de trabajo
display(
    df.groupBy("job")
    .agg(
        count("*").alias("total"),
        sum(when(col("y") == "yes", 1).otherwise(0)).alias("suscritos"),
        round(avg(when(col("y") == "yes", 1).otherwise(0)) * 100, 2).alias("tasa_%")
    )
    .orderBy("tasa_%", ascending=False)
)



job,total,suscritos,tasa_%
student,875,275,31.43
retired,1720,434,25.23
unemployed,1014,144,14.2
admin.,10422,1352,12.97
management,2924,328,11.22
unknown,330,37,11.21
technician,6743,730,10.83
self-employed,1421,149,10.49
housemaid,1060,106,10.0
entrepreneur,1456,124,8.52


> **Hallazgo:** Estudiantes (31%) y jubilados (25%) tienen la mayor tasa de conversión,
> casi 3x el promedio. Blue-collar tiene la menor (6.9%).

In [0]:
# Por mes
display(
    df.groupBy("month")
    .agg(
        count("*").alias("total"),
        round(avg(when(col("y") == "yes", 1).otherwise(0)) * 100, 2).alias("tasa_%")
    )
    .orderBy("tasa_%", ascending=False)
)



month,total,tasa_%
mar,546,50.55
dec,182,48.9
sep,570,44.91
oct,718,43.87
apr,2632,20.48
aug,6178,10.6
jun,5318,10.51
nov,4101,10.14
jul,7174,9.05
may,13769,6.43


> **Hallazgo:** Marzo, diciembre y septiembre tienen tasas ~45-50%, pero volúmenes bajos.
> Mayo concentra el mayor volumen (13.769 llamadas) pero solo 6.4% de conversión.

## 5. Impacto del Contacto Previo

In [0]:
# pdays = 999 significa "nunca contactado" — lo tratamos como null para el análisis
df_pdays = df.withColumn("pdays_clean",
    when(col("pdays") == 999, None).otherwise(col("pdays")))

display(
    df_pdays.groupBy(col("pdays_clean").isNull().alias("nunca_contactado"))
    .agg(
        count("*").alias("total"),
        round(avg(when(col("y") == "yes", 1).otherwise(0)) * 100, 2).alias("tasa_%")
    )
    .orderBy("nunca_contactado")
)



nunca_contactado,total,tasa_%
false,1515,63.83
true,39673,9.26


> **Hallazgo más importante del EDA:**
> Clientes contactados previamente convierten al **63.8%** vs **9.3%** los nuevos.
> Un cliente con contacto previo tiene **7 veces más probabilidad** de suscribir.
>
> **Recomendación de negocio:** agotar lista de clientes con contacto previo antes de llamar nuevos.

## 6. Resumen de Hallazgos EDA

| Hallazgo | Detalle |
|---|---|
| Desbalance de clases | 88.7% no / 11.3% sí — requiere tratamiento |
| Segmento con mayor conversión | Estudiantes (31%) y jubilados (25%) |
| Mejor mes | Marzo, diciembre, septiembre (~45-50%) |
| Variable más discriminante | Contacto previo: 63.8% vs 9.3% |
| Variable con data leakage | `duration` — solo conocida post-llamada, no usar en modelo |